# 📄 Document Analyzer Starter

**AI Learning Playground — Educational Quickstart Blueprint**

Build, run, and deploy a document Q&A system
using a simplified Retrieval-Augmented Generation (RAG) pattern with a local LLM.

> ⚠️ **Prerequisite:** Run **[project-setup.ipynb](project-setup.ipynb)** first to install
> dependencies, validate your GPU, authenticate with Hugging Face, and download all models.

---

## What This Notebook Covers

| Step | Topic | Key Concept |
|------|-------|-------------|
| 1 | Configure Settings | Loading `document.yaml`, resolving model path |
| 2 | Initialize Model | Instantiating `DocumentModel` with LlamaCpp |
| 3 | Demo | Chunk-based RAG: split → per-chunk Q&A → synthesize |
| 4 | GPU Monitoring | VRAM usage during document analysis |
| 5 | Register Model | Logging to MLflow as `AIStudio-EQ-Document` |
| 6 | Verify | Loading registered model and testing on sample text |

## How RAG Works in This Blueprint

```
Document text (any length)
     ↓  Split into 20-line chunks
[
  Chunk 1 → LLM → "Section 1 answer"
  Chunk 2 → LLM → "Not covered in this section."
  Chunk 3 → LLM → "Section 3 answer"
  ...
]
     ↓  Synthesize (filter + combine relevant sections)
Final answer
```

**Why this approach?**
LLMs have a fixed context window. Without chunking, long documents simply cannot be processed.
This implementation is intentionally simple (no vector DB) to keep the focus on the RAG concept.

In [ ]:
import sys
import time

sys.path.insert(0, "..")

start_time = time.time()
print("⏱️  Notebook started")

## 2. Configure Settings

Load `configs/document.yaml` with `capability: document`.
The loader will use this to select `DocumentModel` at serving time.

In [ ]:
import os
from src.utils import load_config

config = load_config("../configs/document.yaml")

model_path     = os.environ.get("MODEL_ARTIFACTS_PATH", config.get("model_path", ""))
context_window = config.get("context_window", 8192)

print(f"Capability : {config.get('capability')}")
print(f"Model path : {model_path}")
print(f"Context    : {context_window} tokens  (~{context_window * 0.75:.0f} words)")

## 3. Verify Assets

In [ ]:
from src.utils import log_asset_status

assets = [
    {"name": "LLM (GGUF)",        "path": model_path,                         "required": True},
    {"name": "Config YAML",       "path": "../configs/document.yaml",          "required": True},
    {"name": "Sample data",       "path": "../data/input/sample_feedback.txt", "required": False},
    {"name": "Document demo UI",  "path": "../demo/document/main.py",          "required": False},
]

log_asset_status(assets)

## 4. Initialize DocumentModel

Instantiate `DocumentModel` — the same class registered in MLflow.
It loads the LLM and accepts an optional `docs_path` for sample document fallback.

In [ ]:
from src.mlflow.models.document import DocumentModel

print("Initializing DocumentModel...")

model = DocumentModel(
    config=config,
    docs_path="../docs",    # Fallback documents directory
    model_path=model_path,
)

print("\n✅ DocumentModel ready")
print(f"   LLM loaded : {'yes' if model.llm is not None else 'no (check model_path)'}")

## 5. Demo: Document Q&A

Call `model.predict()` with a `question` and `input_text` (the document).

The `DocumentModel` will:
1. Split `input_text` into 20-line chunks
2. Run the LLM independently on each chunk
3. Filter out "Not covered" responses
4. Synthesize a final answer from the relevant chunks

In [ ]:
import pandas as pd

# Load the sample document
sample_path = "../data/input/sample_feedback.txt"
with open(sample_path, "r", encoding="utf-8") as f:
    sample_text = f.read()

print(f"Document: {sample_path}")
print(f"Length  : {len(sample_text):,} characters, {len(sample_text.split(chr(10)))} lines")
print(f"Preview : {sample_text[:200]}...")

In [ ]:
result = model.predict(pd.DataFrame([{
    "question":   "What are the main themes and key issues discussed in this document?",
    "input_text": sample_text,
}]))

print("Question: What are the main themes and key issues?")
print("\n" + "─" * 60)
print(result["answer"].iloc[0])

In [ ]:
# Different questions on the same document
questions = [
    "What specific recommendations or action items are mentioned?",
    "What sentiment (positive/negative/neutral) does this document express overall?",
]

for q in questions:
    res = model.predict(pd.DataFrame([{"question": q, "input_text": sample_text}]))
    print(f"Q: {q}")
    print(f"A: {res['answer'].iloc[0][:300]}...")
    print()

In [ ]:
import time
import plotly.graph_objects as go

# Measure analysis time for different document sizes
doc_sizes = {
    "Short (1 chunk)": sample_text[:400],
    "Medium (3 chunks)": sample_text[:1200],
    "Full document":    sample_text,
}

times      = []
chunk_counts = []

for label, text in doc_sizes.items():
    lines = len(text.split("\n"))
    chunk_counts.append(max(1, lines // 20))
    t0 = time.time()
    model.predict(pd.DataFrame([{"question": "Summarize this.", "input_text": text}]))
    times.append(time.time() - t0)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Analysis time (s)",
    x=list(doc_sizes.keys()),
    y=times,
    marker_color="#0096d6",
    text=[f"{t:.1f}s" for t in times],
    textposition="auto",
))
fig.add_trace(go.Scatter(
    name="Chunks processed",
    x=list(doc_sizes.keys()),
    y=chunk_counts,
    mode="lines+markers",
    yaxis="y2",
    marker_color="#00c2e0",
    line_width=2,
))
fig.update_layout(
    title="Document Analysis: Time vs. Document Size",
    yaxis=dict(title="Analysis time (s)", color="#0096d6"),
    yaxis2=dict(title="Chunks processed", overlaying="y", side="right", color="#00c2e0"),
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

## 6. GPU Monitoring

In [ ]:
from src.gpu_monitor import GPUMonitor

monitor = GPUMonitor()
monitor.display_dashboard()

## 7. Register with MLflow

Register as **`AIStudio-EQ-Document`** — independent from the chatbot and image gen models.
The `capability: document` key in the config tells `loader.py` to select `DocumentModel`.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

ARTIFACT_PATH = "AIStudio-EQ-Document"
MODEL_NAME    = "AIStudio-EQ-Document"

print(f"Artifact path  : {ARTIFACT_PATH}")
print(f"Registered as  : {MODEL_NAME}")

In [ ]:
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# DocumentModel input schema: question + the document text
input_schema = Schema([
    ColSpec("string", "question"),    # What you want to know about the document
    ColSpec("string", "input_text"),  # The document content to analyze
])

output_schema = Schema([
    ColSpec("string", "answer"),    # Synthesized RAG answer
    ColSpec("string", "messages"),  # JSON conversation history
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

print("Input  : question (string), input_text (string)")
print("Output : answer (string), messages (JSON string)")

In [ ]:
from src.mlflow.logger import Logger

with mlflow.start_run(run_name=f"register-{ARTIFACT_PATH}") as run:
    Logger.log_model(
        signature     = signature,
        artifact_path = ARTIFACT_PATH,
        config_path   = "../configs/document.yaml",
        docs_path     = "../docs",
        model_path    = model_path,
        demo_folder   = "../demo/document",
    )
    run_id = run.info.run_id

print(f"✅ Model logged | Run ID: {run_id}")

model_uri = f"runs:/{run_id}/{ARTIFACT_PATH}"
reg = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"✅ Registered  : {MODEL_NAME} v{reg.version}")

## 8. Verify Registration

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri=model_uri)

test_doc  = "Artificial intelligence (AI) is intelligence demonstrated by machines. " * 5
test_result = loaded_model.predict(pd.DataFrame([{
    "question":   "What is the main topic of this text?",
    "input_text": test_doc,
}]))

print("✅ Loaded model response:")
print(test_result["answer"].iloc[0][:300])

In [ ]:
elapsed = time.time() - start_time
print(f"⏱️  Total notebook time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

---

## ✅ What We Accomplished

| Step | Result |
|------|--------|
| Environment | CUDA verified, dependencies installed |
| DocumentModel | Initialized with LlamaCpp |
| Demo | Chunk-based RAG on sample document, multiple questions |
| GPU Monitor | VRAM usage during analysis visible |
| Registration | `AIStudio-EQ-Document` registered in Model Registry |
| Verification | Loaded model analyzed test document |

## Next Steps

- **Explore another capability:** Open `voice-assistant-starter.ipynb`
- **Deploy in AI Studio:** Select `AIStudio-EQ-Document` → Deploy → upload documents in the Streamlit UI
- **Improve RAG quality:** Implement embedding-based retrieval (FAISS + sentence-transformers)
- **Try your own documents:** Pass any `.txt` file content as `input_text`